# 04 — Líneas base y comparación final

Ejecuta primero los notebooks 01, 02 y 03 con el mismo valor de `SPLIT` y sube aquí sus archivos `*_predicciones_test.csv`. Este notebook calcula las tres líneas base de popularidad, las métricas por fila y por episodio con IC95 % bootstrap (2,000 remuestreos por conglomerado de episodio), las tablas 2×2 con McNemar exacto y corrección de Holm, y las diferencias con IC95 % para Hit@1, Hit@3, Hit@5 y MRR (Tablas 2 y 3 del artículo).

In [ ]:
!pip -q install statsmodels scikit-learn pandas numpy matplotlib

In [ ]:
import io, os, json, random, warnings
import numpy as np, pandas as pd
from google.colab import files
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, KFold, RandomizedSearchCV, train_test_split
warnings.filterwarnings("ignore")
RS = 42
random.seed(RS); np.random.seed(RS)
SPLIT = 'fila'   # 'fila' = análisis principal del artículo | 'grupo' = sensibilidad agrupada por episodio
FEATURES = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN', 'AFEC']
TARGET = 'MEDICAMENTOS'
KS = (1, 3, 5)
LOG = {}
print("Sube Tacotalpa2024.csv")
up = files.upload(); CSV = next(iter(up))
OUT = f"salida_{SPLIT}"
os.makedirs(OUT, exist_ok=True)

## Preparación común (idéntica en los cuatro notebooks)
Limpieza, episodio aproximado, descripción de clases y partición 80/20 (semilla 42). Con `SPLIT='fila'` se reproduce la partición del artículo (6,277 / 1,560); con `SPLIT='grupo'`, la partición agrupada por episodio usada como análisis de sensibilidad.

In [ ]:
# ---------------------------------------------------------------- 1. Flujo de datos
df = pd.read_csv(CSV, encoding="latin1")
flow = [("Registros en el archivo integrado (3 hospitales, 2020-2022)", len(df))]
for c in ['AFECPRIN', 'AFEC', 'MEDICAMENTOS', 'DESCRIPCION', 'SEXO', 'TIPOURGENCIA']:
    df[c] = df[c].astype("string").str.strip()
df['EDAD'] = pd.to_numeric(df['EDAD'], errors='coerce')
n0 = len(df)
df = df.dropna(subset=FEATURES + [TARGET]).copy()
flow.append(("Excluidos por valores faltantes en variables esenciales", -(n0 - len(df))))
# Episodio aproximado: filas consecutivas con la misma EDAD, SEXO, TIPOURGENCIA y AFECPRIN (el archivo
# integrado conserva el orden de la unión URGENCIAS-AFECCIONES-MEDICAMENTOS, pero no el ID original).
# AFEC no forma parte de la clave porque la unión genera el producto AFEC x MEDICAMENTO dentro de un episodio.
EP_KEY = ['EDAD', 'SEXO', 'TIPOURGENCIA', 'AFECPRIN']
key = df[EP_KEY].astype(str).agg('|'.join, axis=1)
df['EPISODIO'] = (key != key.shift()).cumsum().astype(int)
ndup = int(df.duplicated(subset=FEATURES + [TARGET, 'DESCRIPCION']).sum())
df = df.drop_duplicates(subset=FEATURES + [TARGET, 'DESCRIPCION']).reset_index(drop=True)
flow.append(("Excluidos por duplicados exactos", -ndup))
# Un medicamento repetido dentro de un mismo episodio (distinta presentación) se cuenta una vez
# para el nivel episodio; a nivel fila se conserva como en el protocolo original.
flow.append(("Conjunto analítico (filas episodio-medicamento)", len(df)))
LOG['n_analitico'] = len(df)
LOG['n_episodios'] = int(df.EPISODIO.nunique())
LOG['n_perfiles_unicos'] = int(df[FEATURES].astype(str).agg('|'.join, axis=1).nunique())
ep_size = df.groupby('EPISODIO')[TARGET].nunique()
LOG['episodios_con_1_med'] = int((ep_size == 1).sum())
LOG['episodios_con_2plus_med'] = int((ep_size >= 2).sum())
LOG['filas_en_episodios_multi'] = int(df.EPISODIO.map(df.groupby('EPISODIO').size()).gt(1).sum())
LOG['episodios_multi_afec'] = int((df.groupby('EPISODIO').AFEC.nunique() > 1).sum())
LOG['max_meds_episodio'] = int(ep_size.max())

# ---------------------------------------------------------------- 2. Descripción de clases
vc = df[TARGET].value_counts()
LOG['n_clases'] = int(len(vc))
LOG['clases_1_obs'] = int((vc == 1).sum())
LOG['clases_lt5'] = int((vc < 5).sum())
LOG['clases_lt10'] = int((vc < 10).sum())
for k in (1, 3, 5, 10, 20):
    LOG[f'prop_top{k}'] = float(vc.head(k).sum() / len(df))
p = vc / vc.sum()
LOG['entropia_bits'] = float(-(p * np.log2(p)).sum())
LOG['perplejidad'] = float(2 ** LOG['entropia_bits'])
vc.rename_axis('Medicamento').reset_index(name='Frecuencia').assign(
    Porcentaje=lambda d: 100 * d.Frecuencia / len(df),
    Acumulado=lambda d: d.Porcentaje.cumsum()).to_csv(f"{OUT}/distribucion_medicamentos.csv", index=False)

# ---------------------------------------------------------------- 3. Partición
singleton = vc[vc < 2].index
ep_single = set(df.loc[df[TARGET].isin(singleton), 'EPISODIO'])
if SPLIT == 'grupo':
    pool = df[~df.EPISODIO.isin(ep_single)]
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RS)
    tr_idx, te_idx = next(sgkf.split(pool, pool[TARGET], groups=pool.EPISODIO))
    test_df = pool.iloc[te_idx].copy()
    train_df = pd.concat([pool.iloc[tr_idx], df[df.EPISODIO.isin(ep_single)]]).copy()
else:  # protocolo original: estratificado por fila, sin agrupar
    df_single = df[df[TARGET].isin(singleton)]
    df_split = df[~df[TARGET].isin(singleton)]
    tr, test_df = train_test_split(df_split, test_size=.2, random_state=RS, stratify=df_split[TARGET])
    train_df = pd.concat([tr, df_single]).copy()
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)
LOG['split'] = SPLIT
LOG['n_train'] = len(train_df); LOG['n_test'] = len(test_df)
LOG['ep_train'] = int(train_df.EPISODIO.nunique()); LOG['ep_test'] = int(test_df.EPISODIO.nunique())
LOG['ep_compartidos'] = len(set(train_df.EPISODIO) & set(test_df.EPISODIO))
LOG['filas_singleton_train'] = int(df[TARGET].isin(singleton).sum())
LOG['filas_en_ep_singleton_train'] = int(df.EPISODIO.isin(ep_single).sum())
LOG['clases_test_no_vistas'] = int((~test_df[TARGET].isin(set(train_df[TARGET]))).sum())
flow.append(("Entrenamiento (filas)", len(train_df)))
flow.append(("Prueba interna reservada (filas)", len(test_df)))
pd.DataFrame(flow, columns=['Etapa', 'Registros']).to_csv(f"{OUT}/flujo_datos.csv", index=False)

# Representación a nivel episodio para la prueba: un perfil y el conjunto de medicamentos.
test_ep = test_df.groupby('EPISODIO').agg(**{c: (c, 'first') for c in FEATURES},
                                           MEDS=(TARGET, lambda s: set(s))).reset_index()
LOG['test_episodios'] = len(test_ep)

In [ ]:
# ---------------------------------------------------------------- utilidades de métricas
def row_hits(y, recs):
    """Matriz por caso: rango (1..5) del medicamento verdadero o 0 si no aparece."""
    return np.array([(r.index(t) + 1) if t in r else 0 for t, r in zip(y, recs)])


def set_ranks(sets, recs):
    """Nivel episodio: rango del primer medicamento relevante del episodio."""
    out = []
    for s, r in zip(sets, recs):
        pos = [i + 1 for i, m in enumerate(r) if m in s]
        out.append(pos[0] if pos else 0)
    return np.array(out)


def summarize(ranks, recs, sets=None):
    d = {f'Hit@{k}': float(np.mean((ranks > 0) & (ranks <= k))) for k in KS}
    d['MRR'] = float(np.mean(np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0)))
    d['Cobertura'] = float(np.mean([len(r) > 0 for r in recs]))
    if sets is not None:  # recall@5 a nivel episodio
        d['Recall@5'] = float(np.mean([len(set(r[:5]) & s) / len(s) for s, r in zip(sets, recs)]))
    return d

## Líneas base de popularidad (construidas solo con Entrenamiento)

In [ ]:
# ---------------------------------------------------------------- 4. Líneas base
glob = train_df[TARGET].value_counts().index.tolist()


def cond_pop(cols, min_n=1):
    tab = train_df.groupby(cols + [TARGET]).size().rename('n').reset_index()
    tab = tab.sort_values(cols + ['n', TARGET], ascending=[True] * len(cols) + [False, True])
    return {k if isinstance(k, tuple) else (k,): (g[TARGET].tolist(), int(g.n.sum())) for k, g in tab.groupby(cols)}


pop_dx = cond_pop(['AFECPRIN'])
pop_dx2 = cond_pop(['AFECPRIN', 'AFEC'])


MIN_N = 5  # mínimo de casos de entrenamiento para usar un perfil condicionado; si no, se retrocede


def rec_backoff(row, tables, k=5):
    out = []
    for cols, tab in tables:
        lst, n = tab.get(tuple(row[c] for c in cols), ([], 0))
        if n < MIN_N:
            continue
        for m in lst:
            if m not in out:
                out.append(m)
            if len(out) == k:
                return out
    for m in glob:
        if m not in out:
            out.append(m)
        if len(out) == k:
            return out
    return out


RECS = {}
def predict_all(fn):
    rows = [fn(r) for _, r in test_df.iterrows()]
    eps = [fn(r) for _, r in test_ep.iterrows()]
    return rows, eps

RECS['Popularidad global'] = predict_all(lambda r: glob[:5])
RECS['Popularidad por diagnóstico'] = predict_all(lambda r: rec_backoff(r, [(['AFECPRIN'], pop_dx)]))
RECS['Popularidad por diagnóstico+comorbilidad'] = predict_all(
    lambda r: rec_backoff(r, [(['AFECPRIN', 'AFEC'], pop_dx2), (['AFECPRIN'], pop_dx)]))
LOG['test_dx_no_visto'] = int((~test_df.AFECPRIN.isin(set(train_df.AFECPRIN))).sum())

## Predicciones de las tres técnicas

In [ ]:
print("Sube arbol_predicciones_test.csv, apriori_predicciones_test.csv y kmeans_predicciones_test.csv")
up2 = files.upload()
def load_pred(fragment):
    name = next(n for n in up2 if fragment in n)
    p = pd.read_csv(io.BytesIO(up2[name]), keep_default_na=False)
    if len(p) != len(test_df) or not (p['Real'].astype(str).values == test_df[TARGET].astype(str).values).all():
        raise ValueError(f"{name}: los casos de prueba no coinciden; usa el mismo valor de SPLIT en todos los notebooks.")
    rows = [[x.strip() for x in s.split('|') if x.strip()] for s in p['Top5']]
    first = {}
    for e, r in zip(test_df.EPISODIO, rows):
        first.setdefault(e, r)
    return rows, [first[e] for e in test_ep.EPISODIO]
RECS['Árbol de decisión'] = load_pred('arbol_predicciones')
RECS['K-means'] = load_pred('kmeans_predicciones')
RECS['Apriori'] = load_pred('apriori_predicciones')
print({m: len(r[0]) for m, r in RECS.items()})

## Métricas, intervalos de confianza y comparaciones pareadas

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------- 8. Resultados
y = test_df[TARGET].tolist(); sets = test_ep.MEDS.tolist()
RANK_ROW = {m: row_hits(y, RECS[m][0]) for m in RECS}
RANK_EP = {m: set_ranks(sets, RECS[m][1]) for m in RECS}
groups_row = test_df.EPISODIO.values
B = 2000
rng = np.random.default_rng(RS)
ug = np.unique(groups_row); gi = {g: np.where(groups_row == g)[0] for g in ug}
BOOT_ROW = [np.concatenate([gi[g] for g in rng.choice(ug, len(ug))]) for _ in range(B)]  # bootstrap por conglomerado
n_ep = len(test_ep)
BOOT_EP = [rng.integers(0, n_ep, n_ep) for _ in range(B)]


def metric_vec(ranks):
    return {**{f'Hit@{k}': ((ranks > 0) & (ranks <= k)).astype(float) for k in KS},
            'MRR': np.where(ranks > 0, 1 / np.maximum(ranks, 1), 0.0)}


def table(level):
    RK, RC, BO = (RANK_ROW, 0, BOOT_ROW) if level == 'fila' else (RANK_EP, 1, BOOT_EP)
    rows = []
    for m in RECS:
        base = summarize(RK[m], RECS[m][RC], sets if level == 'episodio' else None)
        mv = metric_vec(RK[m])
        for met, v in mv.items():
            bs = [v[ix].mean() for ix in BO]
            base[met + '_IC95'] = f"[{100*np.quantile(bs,.025):.2f}, {100*np.quantile(bs,.975):.2f}]" if met != 'MRR' \
                else f"[{np.quantile(bs,.025):.4f}, {np.quantile(bs,.975):.4f}]"
        cov = np.array([len(r) > 0 for r in RECS[m][RC]])
        if cov.mean() < 1:
            for k in KS:
                base[f'Hit@{k}_cubiertos'] = float(((RK[m] > 0) & (RK[m] <= k))[cov].mean())
            base['MRR_cubiertos'] = float(mv['MRR'][cov].mean())
        rows.append({'Modelo': m, 'n': len(RK[m]), **base})
    return pd.DataFrame(rows)


res_row = table('fila'); res_ep = table('episodio')
res_row.to_csv(f"{OUT}/resultados_nivel_fila.csv", index=False)
res_ep.to_csv(f"{OUT}/resultados_nivel_episodio.csv", index=False)

# Distribución de longitud de lista Apriori
for lvl, RC in (('fila', 0), ('episodio', 1)):
    ln = pd.Series([len(r) for r in RECS['Apriori'][RC]]).value_counts().sort_index()
    pd.DataFrame({'longitud_lista': ln.index, 'casos': ln.values, 'porcentaje': 100 * ln.values / ln.sum()}).to_csv(
        f"{OUT}/apriori_longitud_lista_{lvl}.csv", index=False)

# Comparaciones pareadas: McNemar exacto (Hit@1) + diferencias con IC bootstrap para todas las métricas
pairs = [('Árbol de decisión', x) for x in ['K-means', 'Apriori', 'Popularidad global', 'Popularidad por diagnóstico',
                                            'Popularidad por diagnóstico+comorbilidad']] + [('K-means', 'Apriori'), ('K-means', 'Popularidad global')]
# Familia de Holm: 7 pares x 3 métricas Hit@K = 21 pruebas McNemar


def paired(level):
    RK, BO = (RANK_ROW, BOOT_ROW) if level == 'fila' else (RANK_EP, BOOT_EP)
    out = []
    for a, b in pairs:
        va, vb = metric_vec(RK[a]), metric_vec(RK[b])
        for k in KS:
            ha, hb = va[f'Hit@{k}'].astype(int), vb[f'Hit@{k}'].astype(int)
            t = [[int(((ha == 1) & (hb == 1)).sum()), int(((ha == 1) & (hb == 0)).sum())],
                 [int(((ha == 0) & (hb == 1)).sum()), int(((ha == 0) & (hb == 0)).sum())]]
            p = mcnemar(t, exact=True).pvalue
            d = va[f'Hit@{k}'] - vb[f'Hit@{k}']
            bs = [d[ix].mean() for ix in BO]
            out.append({'A': a, 'B': b, 'Métrica': f'Hit@{k}', 'Ambos aciertan': t[0][0], 'Sólo A': t[0][1],
                        'Sólo B': t[1][0], 'Ninguno': t[1][1], 'Diferencia (pp)': 100 * d.mean(),
                        'IC95 inf (pp)': 100 * np.quantile(bs, .025), 'IC95 sup (pp)': 100 * np.quantile(bs, .975),
                        'OR discordante (b/c)': (t[0][1] / t[1][0]) if t[1][0] else np.inf, 'p_McNemar': p})
        d = va['MRR'] - vb['MRR']; bs = [d[ix].mean() for ix in BO]
        out.append({'A': a, 'B': b, 'Métrica': 'MRR', 'Diferencia (pp)': d.mean(), 'IC95 inf (pp)': np.quantile(bs, .025),
                    'IC95 sup (pp)': np.quantile(bs, .975),
                    'p_McNemar': np.nan})
    o = pd.DataFrame(out)
    m = o.p_McNemar.notna()
    o.loc[m, 'p_Holm'] = multipletests(o.loc[m, 'p_McNemar'], method='holm')[1]
    return o


paired('fila').to_csv(f"{OUT}/comparaciones_pareadas_fila.csv", index=False)
paired('episodio').to_csv(f"{OUT}/comparaciones_pareadas_episodio.csv", index=False)

# Predicciones
pd.DataFrame({'EPISODIO': test_df.EPISODIO, 'Real': y,
              **{f'Top5_{m}': [' | '.join(r) for r in RECS[m][0]] for m in RECS}}).to_csv(f"{OUT}/predicciones_test_fila.csv", index=False)
with open(f"{OUT}/resumen.json", 'w', encoding='utf-8') as f:
    json.dump(LOG, f, ensure_ascii=False, indent=2, default=float)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)


## Tablas

In [ ]:
display(pd.read_csv(f"{OUT}/flujo_datos.csv"))
display(res_row.round(4)); display(res_ep.round(4))
display(pd.read_csv(f"{OUT}/comparaciones_pareadas_fila.csv").round(4))

## Figura y descarga

In [ ]:
import matplotlib.pyplot as plt
order = ['Árbol de decisión', 'K-means', 'Apriori', 'Popularidad global', 'Popularidad por diagnóstico']
fig, ax = plt.subplots(figsize=(10, 5.2)); w = .15; x = np.arange(3); mets = ['Hit@1', 'Hit@3', 'Hit@5']
for i, m in enumerate(order):
    v = [100*res_row.set_index('Modelo').loc[m, k] for k in mets]
    b = ax.bar(x + (i-2)*w, v, w, label=m, hatch='//' if m.startswith('Popularidad') else '', edgecolor='white')
    ax.bar_label(b, fmt='%.2f%%', fontsize=7, padding=2)
ax.set_xticks(x); ax.set_xticklabels(mets); ax.set_ylabel('Casos con recuperación del medicamento histórico (%)')
ax.set_title('Comparación del rendimiento Top-K en el conjunto interno de prueba'); ax.legend(fontsize=8, ncol=3)
ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.savefig(f"{OUT}/figura_resultados.png", dpi=300); plt.show()
import shutil; shutil.make_archive(OUT, 'zip', OUT); files.download(OUT + '.zip')